# Sistema Peer-to-Peer (P2P) en Python

Este notebook implementa un sistema P2P sencillo que permite que múltiples "peers" se comuniquen entre sí sin un servidor central.

- Cada nodo puede enviar y recibir mensajes.
- Utilizamos `socket` para red.
- Utilizamos `threading` para concurrencia.
- Introduciremos un error de concurrencia de forma intencional, lo analizaremos y lo corregiremos.


## Clase `Peer`
Esta clase representa un nodo P2P.

- Acepta conexiones entrantes.
- Conecta con otros peers conocidos.
- Puede enviar y recibir mensajes. (cliente y server)

In [ ]:
import socket
import threading
import sys

class Peer:
    def __init__(self, username, host, port, known_peers=None):
        self.username = username
        self.host = host
        self.port = port
        self.peers = known_peers if known_peers else []
        self.running = True

        self.connections = []  # Guardamos conexiones salientes para usarlas después

        # Hilo para escuchar conexiones entrantes
        self.server_thread = threading.Thread(target=self.listen_for_peers)
        self.server_thread.start()

        # Conectarse a los peers conocidos
        self.connect_to_known_peers()

    def listen_for_peers(self):
        server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        server.bind((self.host, self.port))
        server.listen()
        print(f"[{self.username}] Listening on {self.host}:{self.port}...")

        while self.running:
            conn, addr = server.accept()
            threading.Thread(target=self.handle_peer, args=(conn, addr)).start()

    def handle_peer(self, conn, addr):
        while self.running:
            try:
                data = conn.recv(1024)
                if not data:
                    break
                print(f"\n[{addr[0]}:{addr[1]}] {data.decode()}")
            except:
                break
        conn.close()

    def connect_to_known_peers(self):
        for peer_addr in self.peers:
            try:
                conn = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
                conn.connect(peer_addr)
                self.connections.append(conn)
                threading.Thread(target=self.send_messages, args=(conn,)).start()
            except Exception as e:
                print(f"No se pudo conectar a {peer_addr}: {e}")

    def send_messages(self, conn):
        while self.running:
            msg = input()
            full_msg = f"{self.username}: {msg}"
            try:
                conn.send(full_msg.encode())
            except:
                break

    def stop(self):
        self.running = False


## Ejecutar varios peers
Puedes probar este sistema ejecutando varias instancias del programa en terminales distintas con distintos puertos:

```bash
# Terminal 1
python p2p_chat.py Alice 5000

# Terminal 2
python p2p_chat.py Bob 5001 127.0.0.1:5000

# Terminal 3
python p2p_chat.py Carol 5002 127.0.0.1:5000 127.0.0.1:5001
```

> En notebooks no es sencillo emular múltiples terminales, así que debes ejecutar el script directamente desde línea de comandos.

## Error de concurrencia intencional
Vamos a agregar un error: varios hilos acceden a `self.connections` sin sincronización.
Esto puede causar errores como `RuntimeError: list changed size during iteration`.

In [4]:
# Ejemplo de acceso concurrente incorrecto
def broadcast_to_all(self, message):
    for conn in self.connections:  # <- Esto puede fallar si otro hilo modifica la lista
        try:
            conn.send(message.encode())
        except:
            continue

## Solución: usar un Lock
Protegemos el acceso a `self.connections` usando `threading.Lock`.

In [5]:
# Arreglo con Lock
import threading

class SafePeer(Peer):
    def __init__(self, *args, **kwargs):
        self.lock = threading.Lock()
        super().__init__(*args, **kwargs)

    def broadcast_to_all(self, message):
        with self.lock:
            for conn in self.connections:
                try:
                    conn.send(message.encode())
                except:
                    continue